# Composite DNA Decoder — Training & Evaluation (3-Base Eta Alphabet)

**80-class alphabet** (4 pure bases + 76 three-base mixtures on a $\pm 2\eta$ grid around 1/3). Theoretical capacity $\log_2(80) \approx 6.32$ bits/position — substantially more than the 34-class 2-mix alphabet (5.09 bits).

**Two run modes** (set `RUN_MODE` in Cell 2):
- `"train_and_eval"` — trains a Bi-LSTM for each coverage level M, saves best weights, then evaluates all four decoders.
- `"eval_only"` — loads previously-saved `best_model_M{M}.pth` and reproduces the same train/val split, then evaluates.

**Single run can sweep one profile or all eight.** Set `ERROR_MODEL` to a name like `"EZ17"`, a list like `["EZ17","R21"]`, or the string `"ALL"`.

**Recommended single-point configuration for the comparison:** `ERROR_MODEL = "EZ17"`, `COVERAGE_LEVELS = [25]` — the cleanest channel (EZ17) and the highest available coverage (M = 25) give the alphabet's extra capacity its best chance to be recoverable.

**Metrics**: NER (nucleotide error rate, BER-analog), SER (sequence error rate), FailureRate (= SER), Accuracy(%). Outputs go to `Results/<auto-named subdir>/` and include `experiment_results.mat`, `experiment_results.txt`, per-M weights, deterministic train/val split indices, and a comparison plot.

In [1]:
# =============================================================================
# CELL 1: IMPORTS & DEVICE
# =============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import random, pickle, json, time, hashlib
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from datetime import datetime
import itertools

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from scipy.io import savemat

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


Using device: cuda
   GPU: NVIDIA GeForce RTX 3080


In [2]:
# =============================================================================
# CELL 2: CONFIGURATION  (training, 3-base eta alphabet)
# =============================================================================

# ------------------- SELECT PROFILE(S) -------------------
# Recommended for the comparison study: "EZ17"  (cleanest channel)
ERROR_MODEL = "EZ17"  # e.g. "EZ17", ["EZ17","R21"], or "ALL"

# ------------------- 3-BASE ETA ALPHABET -------------------
ETA = 0.1
N_STEPS_EACH_SIDE = 2   # gives 5-point grid per active base => 80-class alphabet

# ------------------- RUN MODE -------------------
RUN_MODE = "train_and_eval"  # "train_and_eval" or "eval_only"

# ------------------- COVERAGE LEVELS -------------------
# Recommended single point: just M = 25 (matches MAX_COVERAGE).
# You can add more (e.g. [25, 15]) if you want a small sweep.
COVERAGE_LEVELS = [50]

# ------------------- KL/ML SMOOTHING EPSILON -------------------
# Three options:
#   "auto"   -> use per-profile best epsilon from ERROR_MODEL_SPECS (recommended)
#   number   -> use the same epsilon for every profile
#   dict     -> per-profile override; missing keys fall back to "auto"
EPSILON_KLML = "auto"

# ------------------- DATASET PARAMETERS (must match generator) -------------------
NUM_SAMPLES   = 100000
MAX_COVERAGE  = 50
DATASET_DIR   = "./dataset"

# ------------------- TRAINING HYPERPARAMETERS -------------------
HIDDEN_DIM     = 128
NUM_LAYERS     = 2
DROPOUT        = 0.2
BIDIRECTIONAL  = True
BATCH_SIZE     = 500
LEARNING_RATE  = 1e-3
WEIGHT_DECAY   = 1e-4
EPOCHS         = 100
PATIENCE       = 10
WARMUP_EPOCHS  = 10
MIN_LR         = 1e-6
TRAIN_FRACTION = 0.8
SEED           = 42

RESULTS_ROOT   = "./Results"

# ------------------- PROFILE SPECS (with per-profile best epsilon) -------------------
ERROR_MODEL_SPECS = {
    "EZ17":  {"seq_length": 136, "name": "EZ17",  "platform": "Illumina miSeq + Twist Bioscience",
              "best_eps_klml": 0.01},
    "G15":   {"seq_length": 104, "name": "G15",   "platform": "Illumina miSeq + CustomArray",
              "best_eps_klml": 0.05},
    "O17":   {"seq_length": 77,  "name": "O17",   "platform": "Illumina NextSeq + Twist Bioscience",
              "best_eps_klml": 0.01},
    "R21":   {"seq_length": 136, "name": "R21",   "platform": "Nanopore MinION + Twist Bioscience",
              "best_eps_klml": 0.1},
    "B22":   {"seq_length": 136, "name": "B22",   "platform": "Nanopore MinION short-read + Twist",
              "best_eps_klml": 0.1},
    "BOS22": {"seq_length": 136, "name": "BOS22", "platform": "Illumina miSeq 2022 + Twist",
              "best_eps_klml": 0.001},
    "NP22":  {"seq_length": 136, "name": "NP22",  "platform": "Nanopore Pilot Nov-2022",
              "best_eps_klml": 0.1},
    "NPF22": {"seq_length": 136, "name": "NPF22", "platform": "Nanopore Full Nov-2022",
              "best_eps_klml": 0.1},
    "erlich":   {"seq_length": 136, "name": "EZ17",  "platform": "Illumina miSeq + Twist Bioscience",
                 "best_eps_klml": 0.01},
    "grass":    {"seq_length": 104, "name": "G15",   "platform": "Illumina miSeq + CustomArray",
                 "best_eps_klml": 0.05},
    "organick": {"seq_length": 77,  "name": "O17",   "platform": "Illumina NextSeq + Twist Bioscience",
                 "best_eps_klml": 0.01},
}

ALL_PROFILES = ["EZ17", "G15", "O17", "R21", "B22", "BOS22", "NP22", "NPF22"]
ALIASES = {"erlich": "EZ17", "grass": "G15", "organick": "O17"}
def _canon(p): return ALIASES.get(p, p)
def _resolve_profiles(spec):
    if isinstance(spec, str):
        if spec.upper() == "ALL":
            return list(ALL_PROFILES)
        return [_canon(spec)]
    elif isinstance(spec, (list, tuple)):
        return [_canon(s) for s in spec]
    raise TypeError(f"ERROR_MODEL must be str or list, got {type(spec)}")

PROFILES_TO_RUN = _resolve_profiles(ERROR_MODEL)
for p in PROFILES_TO_RUN:
    assert p in ERROR_MODEL_SPECS, f"Unknown profile {p}"
assert RUN_MODE in ("train_and_eval", "eval_only"), f"Unknown RUN_MODE: {RUN_MODE}"

def _resolve_epsilon_for(profile_name):
    specs = ERROR_MODEL_SPECS[profile_name]
    profile_best = specs.get("best_eps_klml", 0.01)
    if isinstance(EPSILON_KLML, str):
        if EPSILON_KLML.lower() == "auto":
            return profile_best
        return float(EPSILON_KLML)
    elif isinstance(EPSILON_KLML, dict):
        return float(EPSILON_KLML.get(profile_name, profile_best))
    elif isinstance(EPSILON_KLML, (int, float)):
        return float(EPSILON_KLML)
    raise TypeError(f"EPSILON_KLML bad type: {type(EPSILON_KLML)}")

def make_config_for_profile(profile_name):
    specs = ERROR_MODEL_SPECS[profile_name]
    eps = _resolve_epsilon_for(profile_name)
    n_grid = 2 * N_STEPS_EACH_SIDE + 1
    alphabet_mode = f"eta3mix_{ETA}_n{n_grid}"
    subdir = (f"{specs['name']}_{alphabet_mode}"
              f"_H{HIDDEN_DIM}_L{NUM_LAYERS}_lr{LEARNING_RATE:.0e}"
              f"_eps{eps:g}_seed{SEED}")
    results_dir = os.path.join(RESULTS_ROOT, subdir)
    dataset_name = f"dna_{specs['name']}_{alphabet_mode}"
    dataset_path = f"{DATASET_DIR}/{dataset_name}_{NUM_SAMPLES}_{MAX_COVERAGE}.pkl"
    # Vocab size is determined entirely by ETA and N_STEPS_EACH_SIDE; we set it
    # by computing the alphabet once below in Cell 4.
    cfg = {
        "error_model":  profile_name,
        "error_name":   specs["name"],
        "platform":     specs["platform"],
        "seq_length":   specs["seq_length"],
        "alphabet_mode": alphabet_mode,
        "vocab_size":   None,    # filled in once Cell 4 runs
        "eta":          ETA,
        "n_steps":      N_STEPS_EACH_SIDE,
        "dataset_path": dataset_path,
        "results_dir":  results_dir,
        "run_mode":     RUN_MODE,
        "coverage_levels": list(COVERAGE_LEVELS),
        "input_channels": 4,
        "hidden_dim":   HIDDEN_DIM,
        "num_layers":   NUM_LAYERS,
        "dropout":      DROPOUT,
        "bidirectional": BIDIRECTIONAL,
        "batch_size":   BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "epochs":       EPOCHS,
        "patience":     PATIENCE,
        "warmup_epochs": WARMUP_EPOCHS,
        "min_lr":       MIN_LR,
        "train_fraction": TRAIN_FRACTION,
        "epsilon_klml": eps,
        "epsilon_klml_source": ("auto (per-profile best)"
                                if (isinstance(EPSILON_KLML, str) and EPSILON_KLML.lower() == "auto")
                                else ("dict override" if isinstance(EPSILON_KLML, dict)
                                      else "scalar override")),
        "seed":         SEED,
    }
    return cfg

os.makedirs(RESULTS_ROOT, exist_ok=True)

print('=' * 72)
print(' CONFIGURATION  (3-base eta alphabet)')
print('=' * 72)
print(f"   Run mode        : {RUN_MODE}")
if len(PROFILES_TO_RUN) == 1:
    print(f"   Mode            : SINGLE profile")
else:
    print(f"   Mode            : BATCH ({len(PROFILES_TO_RUN)} profiles)")
print(f"   Profiles        : {', '.join(PROFILES_TO_RUN)}")
print(f"   Eta             : {ETA}")
print(f"   Grid steps      : ±{N_STEPS_EACH_SIDE} (i.e. {2*N_STEPS_EACH_SIDE+1} values per active base)")
print(f"   Coverage levels : {COVERAGE_LEVELS}")
if isinstance(EPSILON_KLML, str) and EPSILON_KLML.lower() == "auto":
    print(f"   Epsilon (KL/ML) : auto (per-profile best from ERROR_MODEL_SPECS)")
    for p in PROFILES_TO_RUN:
        print(f"       {p:<6} → eps = {_resolve_epsilon_for(p):g}")
else:
    print(f"   Epsilon (KL/ML) : {EPSILON_KLML}")
print(f"   Seed            : {SEED}")
print(f"   Results root    : {RESULTS_ROOT}/")
print()
for p in PROFILES_TO_RUN:
    cfg = make_config_for_profile(p)
    print(f"     • {p:<6} → {cfg['results_dir']}")
print('=' * 72)


 CONFIGURATION  (3-base eta alphabet)
   Run mode        : train_and_eval
   Mode            : SINGLE profile
   Profiles        : EZ17
   Eta             : 0.1
   Grid steps      : ±2 (i.e. 5 values per active base)
   Coverage levels : [50]
   Epsilon (KL/ML) : auto (per-profile best from ERROR_MODEL_SPECS)
       EZ17   → eps = 0.01
   Seed            : 42
   Results root    : ./Results/

     • EZ17   → ./Results/EZ17_eta3mix_0.1_n5_H128_L2_lr1e-03_eps0.01_seed42


In [3]:
# =============================================================================
# CELL 3: SEED HELPERS
# =============================================================================
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def split_seed_for(base_seed, dataset_path, coverage_M):
    key = f"{base_seed}|{os.path.basename(dataset_path)}|M{coverage_M}"
    h = hashlib.sha256(key.encode("utf-8")).digest()
    return int.from_bytes(h[:4], byteorder="big", signed=False)

print("Seed helpers ready.")


Seed helpers ready.


In [4]:
# =============================================================================
# CELL 4: BUILD 3-BASE ETA ALPHABET  (vocab=80 for eta=0.1, n_steps=2)
# =============================================================================

def build_eta_3mix_alphabet(eta, n_steps_each_side=2):
    """Build the 3-base mixture alphabet centered at 1/3 with step `eta`.

    Each non-pure symbol has weights (p_a, p_b, p_c, 0) on three of the four
    bases (the fourth, "inactive", has weight 0). The weights live on the grid
        {1/3 - n*eta, ..., 1/3 - eta, 1/3, 1/3 + eta, ..., 1/3 + n*eta}
    where n = n_steps_each_side. With eta=0.1 and n=2 the grid is
        {0.1333, 0.2333, 0.3333, 0.4333, 0.5333}
    constrained to sum to exactly 1 across the three active bases.

    For each of the 4 choices of inactive base there are 19 valid ordered
    triples on this grid that sum to 1 (with eta=0.1, n_steps=2), giving
    19 * 4 = 76 mixture symbols. Adding the 4 pure bases gives 80 total
    symbols. log2(80) ≈ 6.32 bits/position.

    Returns
    -------
    composite_map: dict[str, list[(base, prob)]]
        Maps symbol name to its component (base, probability) list.
        Pure bases use a single (base, 1.0) entry.
    symbol_to_idx: dict[str, int]
        Stable integer index for each symbol (pure bases occupy indices 0..3).
    ideal_vectors: torch.Tensor of shape (V, 4)
        Row v is the [A, C, G, T] probability vector for symbol with index v.
    """
    import itertools
    third = 1.0 / 3.0
    grid = [third + l * eta for l in range(-n_steps_each_side,
                                           n_steps_each_side + 1)]
    bases = ['A', 'C', 'G', 'T']

    composite_map = {
        'A': [('A', 1.0)],
        'C': [('C', 1.0)],
        'G': [('G', 1.0)],
        'T': [('T', 1.0)],
    }
    symbol_to_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    ideal_vectors_list = [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ]

    cur = 4
    # For deterministic naming, fix inactive base and enumerate active triple
    # in increasing-base order; then enumerate grid triples summing to 1.
    for inactive in bases:
        active = [b for b in bases if b != inactive]   # length 3, in ACGT order
        for triple in itertools.product(grid, repeat=3):
            if abs(sum(triple) - 1.0) < 1e-9:
                p1, p2, p3 = triple
                # Symbol name encodes inactive base and the integer offsets
                # ell_i = (p_i - 1/3)/eta for each active base (in ACGT order).
                ells = []
                for p in triple:
                    ell = int(round((p - third) / eta))
                    ells.append(ell)
                name = f"X{inactive}_" + "_".join(
                    (f"{ell:+d}" if ell != 0 else "0") for ell in ells
                )
                comps = list(zip(active, triple))
                composite_map[name] = comps
                symbol_to_idx[name] = cur
                vec = [0.0, 0.0, 0.0, 0.0]
                for b, p in zip(active, triple):
                    vec[bases.index(b)] = p
                ideal_vectors_list.append(vec)
                cur += 1
    import torch
    ideal_vectors = torch.tensor(ideal_vectors_list, dtype=torch.float32)
    return composite_map, symbol_to_idx, ideal_vectors

def build_symbol_to_idx_and_vectors(cfg):
    cm, sti, vec = build_eta_3mix_alphabet(cfg['eta'], cfg['n_steps'])
    return sti, vec

_cm0, _sti0, _vec0 = build_eta_3mix_alphabet(ETA, N_STEPS_EACH_SIDE)
VOCAB_SIZE = len(_sti0)
print(f"3-base eta alphabet: {VOCAB_SIZE} symbols  ({np.log2(VOCAB_SIZE):.4f} bits/position)")
print(f"   Pure bases  : 4")
print(f"   Mixtures    : {VOCAB_SIZE - 4}")
print(f"   Sample symbols (first 8):")
for sym, idx in list(sorted(_sti0.items(), key=lambda x: x[1]))[:8]:
    v = _vec0[idx].tolist()
    v_str = '[' + ', '.join(f'{x:.4f}' for x in v) + ']'
    print(f"      idx={idx:>3}  {sym:<15}  {v_str}")

# Patch vocab_size into per-profile configs (they were created with vocab_size=None)
for _p in PROFILES_TO_RUN:
    # The CONFIG is rebuilt every time make_config_for_profile is called in Cell 13,
    # so we just need to override the vocab_size in the factory.
    pass
_make_cfg_orig = make_config_for_profile
def make_config_for_profile(name, _orig=_make_cfg_orig, _vsize=VOCAB_SIZE):
    cfg = _orig(name)
    cfg['vocab_size'] = _vsize
    return cfg

3-base eta alphabet: 80 symbols  (6.3219 bits/position)
   Pure bases  : 4
   Mixtures    : 76
   Sample symbols (first 8):
      idx=  0  A                [1.0000, 0.0000, 0.0000, 0.0000]
      idx=  1  C                [0.0000, 1.0000, 0.0000, 0.0000]
      idx=  2  G                [0.0000, 0.0000, 1.0000, 0.0000]
      idx=  3  T                [0.0000, 0.0000, 0.0000, 1.0000]
      idx=  4  XA_-2_0_+2       [0.0000, 0.1333, 0.3333, 0.5333]
      idx=  5  XA_-2_+1_+1      [0.0000, 0.1333, 0.4333, 0.4333]
      idx=  6  XA_-2_+2_0       [0.0000, 0.1333, 0.5333, 0.3333]
      idx=  7  XA_-1_-1_+2      [0.0000, 0.2333, 0.2333, 0.5333]


In [5]:
# =============================================================================
# CELL 6: DATASET CLASS & DETERMINISTIC SPLIT
# =============================================================================
def preprocess_cluster_to_matrix(cluster_reads, target_length):
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    for read in cluster_reads:
        L = len(read)
        if L == 0: continue
        for t in range(target_length):
            r = int((t + 0.5) * (L / target_length))
            if r >= L: r = L - 1
            b = read[r]
            if b in base_map:
                profile_matrix[base_map[b], t] += 1.0
    if num_reads > 0:
        profile_matrix /= num_reads
    return profile_matrix


class CompositeDNADataset(Dataset):
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw = pickle.load(f)
        self.samples = raw['data']
        self.metadata = raw['metadata']
        self.seq_length = seq_length
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        if self.limit_coverage is not None:
            cluster = cluster[:min(self.limit_coverage, len(cluster))]
        x = preprocess_cluster_to_matrix(cluster, self.seq_length)
        y = np.array([self.symbol_to_idx[s] for s in item['label']], dtype=np.longlong)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)


def make_reproducible_split(full_ds, base_seed, dataset_path, coverage_M,
                            train_frac=0.8, split_dir=None):
    n = len(full_ds)
    n_train = int(train_frac * n)
    split_seed = split_seed_for(base_seed, dataset_path, coverage_M)
    split_file = None
    if split_dir is not None:
        os.makedirs(split_dir, exist_ok=True)
        split_file = os.path.join(split_dir, f"split_indices_M{coverage_M}.pkl")
    if split_file is not None and os.path.exists(split_file):
        with open(split_file, 'rb') as f:
            saved = pickle.load(f)
        if saved['seed'] == split_seed and saved['n'] == n:
            print(f"      Loaded saved train/val split from {split_file}")
            return (Subset(full_ds, saved['train_idx']),
                    Subset(full_ds, saved['val_idx']), split_seed)
    rng = np.random.RandomState(split_seed)
    perm = rng.permutation(n)
    train_idx = perm[:n_train].tolist()
    val_idx = perm[n_train:].tolist()
    if split_file is not None:
        with open(split_file, 'wb') as f:
            pickle.dump({'seed': split_seed, 'n': n,
                         'train_idx': train_idx, 'val_idx': val_idx,
                         'base_seed': base_seed, 'coverage_M': coverage_M,
                         'dataset_path': dataset_path}, f)
        print(f"      Saved train/val split to {split_file}")
    return Subset(full_ds, train_idx), Subset(full_ds, val_idx), split_seed


In [6]:
# =============================================================================
# CELL 7: BI-LSTM MODEL
# =============================================================================
class CompositeDecoderLSTM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0.0,
        )
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)


In [7]:
# =============================================================================
# CELL 8: BASELINE DECODERS  (KL / ML use configurable smoothing epsilon)
# =============================================================================
def min_distance_decoder(obs, ideal_vectors):
    """obs:(B,L,4), ideal:(V,4)"""
    dists = torch.sum(
        (obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)

def kl_divergence_decoder(obs, ideal_vectors, epsilon):
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    ce = -(obs.unsqueeze(2) * log_ideal).sum(dim=-1)
    return torch.argmin(ce, dim=-1)

def maximum_likelihood_decoder(obs, ideal_vectors, epsilon):
    ideal_safe = torch.clamp(ideal_vectors.clone(), min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    ll = (obs.unsqueeze(2) * log_ideal).sum(dim=-1)
    return torch.argmax(ll, dim=-1)


In [8]:
# =============================================================================
# CELL 9: EARLY STOPPING & TRAINING LOOP
# =============================================================================
class EarlyStopping:
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience; self.counter = 0
        self.best_score = None; self.early_stop = False
        self.path = path; self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None or score > self.best_score:
            if self.verbose and self.best_val_loss != float('inf'):
                print(f"      Val loss improved ({self.best_val_loss:.4f} -> {val_loss:.4f}). Saving.")
            self.best_score = score
            torch.save(model.state_dict(), self.path)
            self.best_val_loss = val_loss; self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

def train_model(model, train_loader, val_loader, config, weights_path, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(),
                            lr=config['learning_rate'],
                            weight_decay=config['weight_decay'])
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=config['warmup_epochs'])
    # T_max is the number of post-warmup (cosine) epochs. Guard against
    # epochs <= warmup_epochs (e.g. tiny smoke-test configs), which would make
    # CosineAnnealingLR divide by zero.
    _t_max = max(1, config['epochs'] - config['warmup_epochs'])
    cosine = CosineAnnealingLR(optimizer,
                               T_max=_t_max,
                               eta_min=config['min_lr'])
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine],
                             milestones=[config['warmup_epochs']])
    stopper = EarlyStopping(patience=config['patience'], path=weights_path, verbose=True)
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    print(f"   Training: epochs={config['epochs']}, patience={config['patience']}, "
          f"warmup={config['warmup_epochs']}, lr={config['learning_rate']} → {config['min_lr']}")
    for epoch in range(config['epochs']):
        t0 = time.time()
        model.train(); train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            out = model(inputs)
            loss = criterion(out, labels); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        train_loss /= max(len(train_loader), 1)
        model.eval(); val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                val_loss += criterion(model(inputs), labels).item()
        val_loss /= max(len(val_loader), 1)
        cur_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(cur_lr)
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"LR: {cur_lr:.2e} | Time: {time.time()-t0:.1f}s")
        scheduler.step()
        stopper(val_loss, model)
        if stopper.early_stop:
            print(f"   Early stopping at epoch {epoch+1}"); break
    return history


In [9]:
# =============================================================================
# CELL 10: EVALUATION  (NER, SER, FailureRate, Accuracy)
# =============================================================================
def _per_sequence_errors(pred, labels):
    B, L = labels.shape
    mismatch = (pred != labels)
    pos_errors = mismatch.sum().item()
    seq_has_error = mismatch.any(dim=1).sum().item()
    return B, seq_has_error, B * L, pos_errors

def evaluate_all_decoders(model, loader, ideal_vectors, device, epsilon_klml):
    model.eval()
    keys = ['lstm', 'mindist', 'kl', 'ml']
    pos_err = {k: 0 for k in keys}
    pos_tot = {k: 0 for k in keys}
    seq_err = {k: 0 for k in keys}
    seq_tot = {k: 0 for k in keys}
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)
            logits = model(inputs)
            pred_lstm = torch.argmax(logits, dim=1)
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            pred_kl = kl_divergence_decoder(obs, ideal_vectors, epsilon_klml)
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors, epsilon_klml)
            for k, pred in [('lstm', pred_lstm), ('mindist', pred_mindist),
                            ('kl', pred_kl), ('ml', pred_ml)]:
                B, se, P, pe = _per_sequence_errors(pred, labels)
                seq_tot[k] += B; seq_err[k] += se
                pos_tot[k] += P; pos_err[k] += pe
    metrics = {}
    for k in keys:
        ner = pos_err[k] / max(pos_tot[k], 1)
        ser = seq_err[k] / max(seq_tot[k], 1)
        metrics[k] = {
            'accuracy': 100.0 * (1.0 - ner),
            'NER': ner, 'SER': ser, 'failure_rate': ser,
            'positions_correct': pos_tot[k] - pos_err[k],
            'positions_total':   pos_tot[k],
            'sequences_correct': seq_tot[k] - seq_err[k],
            'sequences_total':   seq_tot[k],
        }
    return metrics


In [10]:
# =============================================================================
# CELL 11: PER-COVERAGE EXPERIMENT (train_and_eval OR eval_only)
# =============================================================================
def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors,
                                device, run_mode):
    print(f"\n{'='*72}")
    print(f"  COVERAGE M = {coverage_M}  |  profile = {config['error_name']}"
          f"  |  mode = {run_mode}")
    print(f"  seq_length = {config['seq_length']}, vocab = {config['vocab_size']}, "
          f"alphabet = {config['alphabet_mode']}")
    print(f"{'='*72}")
    set_global_seed(config['seed'])
    full_ds = CompositeDNADataset(config['dataset_path'], config['seq_length'],
                                  symbol_to_idx, limit_coverage=coverage_M)
    print(f"   Dataset loaded: {len(full_ds):,} samples from {config['dataset_path']}")
    train_ds, val_ds, split_seed = make_reproducible_split(
        full_ds, base_seed=config['seed'],
        dataset_path=config['dataset_path'],
        coverage_M=coverage_M, train_frac=config['train_fraction'],
        split_dir=config['results_dir'])
    print(f"   Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  split_seed={split_seed}")
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'],
                              shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'],
                            shuffle=False, num_workers=0)
    model = CompositeDecoderLSTM(config).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Model: {config['vocab_size']}-class Bi-LSTM, {n_params:,} parameters")
    rdir = config['results_dir']; os.makedirs(rdir, exist_ok=True)
    best_path  = os.path.join(rdir, f"best_model_M{coverage_M}.pth")
    final_path = os.path.join(rdir, f"final_model_M{coverage_M}.pth")
    hist_path  = os.path.join(rdir, f"training_history_M{coverage_M}.json")
    history = None
    if run_mode == 'train_and_eval':
        history = train_model(model, train_loader, val_loader, config, best_path, device)
        torch.save(model.state_dict(), final_path)
        with open(hist_path, 'w') as f: json.dump(history, f, indent=2)
        model.load_state_dict(torch.load(best_path, map_location=device))
    elif run_mode == 'eval_only':
        if not os.path.exists(best_path):
            raise FileNotFoundError(
                f"eval_only requires best_model_M{coverage_M}.pth at: {best_path}\n"
                "  Run with RUN_MODE='train_and_eval' first.")
        model.load_state_dict(torch.load(best_path, map_location=device))
        print(f"   Loaded best weights from {best_path}")
    else:
        raise ValueError(f"Unknown run_mode: {run_mode!r}")
    metrics = evaluate_all_decoders(model, val_loader, ideal_vectors, device,
                                    epsilon_klml=config['epsilon_klml'])
    print(f"\n   Results @ M={coverage_M}:")
    print(f"      {'Decoder':<14} {'Acc(%)':>8} {'NER':>10} {'SER':>10} {'FailRate':>10}")
    for k, name in [('lstm','Bi-LSTM'), ('mindist','Min.Dist.'),
                    ('kl','KL Div.'), ('ml','Max Lik.')]:
        m = metrics[k]
        print(f"      {name:<14} {m['accuracy']:>8.4f} {m['NER']:>10.6f} "
              f"{m['SER']:>10.6f} {m['failure_rate']:>10.6f}")
    return metrics, history


In [11]:
# =============================================================================
# CELL 12: SAVE RESULTS  ( .mat  +  .txt  +  comparison plot )
# =============================================================================
def _np(arr):
    return np.array(arr, dtype=np.float64)

def save_results_mat(results, config, mat_path):
    out = {}
    out['coverage_M']    = _np(results['coverage'])
    out['decoder_names'] = np.array(['lstm','mindist','kl','ml'], dtype=object)
    for d in ['lstm','mindist','kl','ml']:
        out[f'Acc_{d}']             = _np([m['accuracy']        for m in results['per_M'][d]])
        out[f'NER_{d}']             = _np([m['NER']             for m in results['per_M'][d]])
        out[f'SER_{d}']             = _np([m['SER']             for m in results['per_M'][d]])
        out[f'FailureRate_{d}']     = _np([m['failure_rate']    for m in results['per_M'][d]])
        out[f'positions_total_{d}'] = _np([m['positions_total'] for m in results['per_M'][d]])
        out[f'sequences_total_{d}'] = _np([m['sequences_total'] for m in results['per_M'][d]])
    meta_str_keys = ['error_model','error_name','alphabet_mode','dataset_path',
                     'results_dir','run_mode','timestamp','epsilon_klml_source','platform']
    for k in meta_str_keys:
        if k in results['config']:
            out[k] = str(results['config'][k])
    meta_num_keys = ['seq_length','vocab_size','hidden_dim','num_layers','dropout',
                     'batch_size','learning_rate','weight_decay','epochs','patience',
                     'warmup_epochs','min_lr','epsilon_klml','seed','train_fraction',
                     'eta','n_steps']
    for k in meta_num_keys:
        if k in results['config']:
            out[k] = results['config'][k]
    savemat(mat_path, out, do_compression=True)
    print(f"   Saved .mat results: {mat_path}")

def build_results_report(results, config):
    """Build the full human-readable experiment report as a single string.

    The exact same text is written to the .txt file AND printed at the end of
    the run, so the on-screen summary is identical to the saved report.
    """
    decoders = [('lstm','Bi-LSTM'), ('mindist','Min. Distance'),
                ('kl','KL Divergence'), ('ml','Max. Likelihood')]
    L = []  # list of lines
    L.append('='*78)
    L.append(' COMPOSITE-DNA DECODER — EXPERIMENT REPORT (3-base eta alphabet)')
    L.append('='*78)
    L.append('')
    for label, key in [
        ('Timestamp',       'timestamp'),
        ('Run mode',        'run_mode'),
        ('Error profile',   'error_name'),
        ('Platform',        'platform'),
        ('Alphabet mode',   'alphabet_mode'),
        ('Eta',             'eta'),
        ('Vocabulary size', 'vocab_size'),
        ('Sequence length', 'seq_length'),
        ('Dataset',         'dataset_path'),
    ]:
        v = results['config'].get(key, 'n/a')
        L.append(f"{label:<17}: {v}")
    L.append('')
    L.append('Hyperparameters')
    L.append('-'*78)
    for k in ['hidden_dim','num_layers','dropout','bidirectional',
              'batch_size','learning_rate','weight_decay','epochs','patience',
              'warmup_epochs','min_lr','epsilon_klml','epsilon_klml_source',
              'seed','train_fraction']:
        if k in results['config']:
            L.append(f"  {k:<22}: {results['config'][k]}")
    L.append('')
    L.append('Coverage levels evaluated')
    L.append('-'*78)
    L.append(f"  M = {results['coverage']}")
    L.append('')
    for d_key, d_name in decoders:
        L.append('')
        L.append(f"  Decoder: {d_name}")
        L.append(f"  {'M':>4} | {'Accuracy(%)':>12} | {'NER':>12} | {'SER':>12} | "
                 f"{'FailureRate':>12} | {'#pos':>10} | {'#seq':>10}")
        L.append('  ' + '-'*82)
        for i, M in enumerate(results['coverage']):
            m = results['per_M'][d_key][i]
            L.append(f"  {M:>4} | {m['accuracy']:>12.4f} | {m['NER']:>12.6f} | "
                     f"{m['SER']:>12.6f} | {m['failure_rate']:>12.6f} | "
                     f"{m['positions_total']:>10} | {m['sequences_total']:>10}")
    L.append('')
    L.append('')
    L.append('Side-by-side comparison (accuracy %, higher is better)')
    L.append('-'*78)
    L.append(f"  {'M':>4} | {'Bi-LSTM':>10} | {'Min.Dist':>10} | {'KL Div':>10} | {'Max Lik':>10}")
    L.append('  ' + '-'*56)
    for i, M in enumerate(results['coverage']):
        L.append(f"  {M:>4} | "
                 f"{results['per_M']['lstm'][i]['accuracy']:>10.4f} | "
                 f"{results['per_M']['mindist'][i]['accuracy']:>10.4f} | "
                 f"{results['per_M']['kl'][i]['accuracy']:>10.4f} | "
                 f"{results['per_M']['ml'][i]['accuracy']:>10.4f}")
    L.append('')
    L.append('')
    L.append('Side-by-side comparison (NER, lower is better)')
    L.append('-'*78)
    L.append(f"  {'M':>4} | {'Bi-LSTM':>12} | {'Min.Dist':>12} | {'KL Div':>12} | {'Max Lik':>12}")
    L.append('  ' + '-'*64)
    for i, M in enumerate(results['coverage']):
        L.append(f"  {M:>4} | "
                 f"{results['per_M']['lstm'][i]['NER']:>12.6f} | "
                 f"{results['per_M']['mindist'][i]['NER']:>12.6f} | "
                 f"{results['per_M']['kl'][i]['NER']:>12.6f} | "
                 f"{results['per_M']['ml'][i]['NER']:>12.6f}")
    L.append('')
    L.append('')
    L.append('Side-by-side comparison (SER / Failure rate, lower is better)')
    L.append('-'*78)
    L.append(f"  {'M':>4} | {'Bi-LSTM':>12} | {'Min.Dist':>12} | {'KL Div':>12} | {'Max Lik':>12}")
    L.append('  ' + '-'*64)
    for i, M in enumerate(results['coverage']):
        L.append(f"  {M:>4} | "
                 f"{results['per_M']['lstm'][i]['SER']:>12.6f} | "
                 f"{results['per_M']['mindist'][i]['SER']:>12.6f} | "
                 f"{results['per_M']['kl'][i]['SER']:>12.6f} | "
                 f"{results['per_M']['ml'][i]['SER']:>12.6f}")
    L.append('')
    L.append('='*78)
    L.append(' END OF REPORT')
    L.append('='*78)
    return '\n'.join(L) + '\n'


def save_results_txt(results, config, txt_path):
    """Write the experiment report to a .txt file AND print it to the notebook."""
    report = build_results_report(results, config)
    with open(txt_path, 'w') as f:
        f.write(report)
    print(f"   Saved .txt report : {txt_path}")
    # Echo the full report to the notebook output so it is visible at run end.
    print()
    print(report)

def plot_comparison(results, save_path, config):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    Ms = results['coverage']
    decoders = [('lstm','Bi-LSTM','#2980b9'), ('mindist','Min. Distance','#7f8c8d'),
                ('kl','KL Divergence','#27ae60'), ('ml','Max. Likelihood','#c0392b')]
    for k, label, color in decoders:
        accs = [m['accuracy'] for m in results['per_M'][k]]
        ners = [m['NER']      for m in results['per_M'][k]]
        ax1.plot(Ms, accs, 'o-', lw=2.2, ms=7, color=color, label=label)
        ax2.plot(Ms, ners, 'o-', lw=2.2, ms=7, color=color, label=label)
    for ax, ylab, title in [(ax1, 'Accuracy (%)', 'Accuracy vs Coverage'),
                            (ax2, 'NER', 'Nucleotide Error Rate vs Coverage')]:
        ax.set_xlabel('Coverage Depth M', fontsize=11)
        ax.set_ylabel(ylab, fontsize=11)
        ax.set_title(f"{title} — {config['error_name']} / 3-base eta",
                     fontsize=12)
        ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    ax1.set_ylim(0, 105); ax2.set_yscale('log')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight'); plt.close()
    print(f"   Saved comparison plot: {save_path}")


In [12]:
# =============================================================================
# CELL 13: MAIN EXECUTION LOOP
# =============================================================================
batch_t0 = time.time()
batch_summary = []

for prof_idx, profile_name in enumerate(PROFILES_TO_RUN, start=1):
    cfg = make_config_for_profile(profile_name)

    print(f"\n\n{'#'*78}")
    print(f"#  [{prof_idx}/{len(PROFILES_TO_RUN)}]  Profile: {profile_name}  ({cfg['platform']})")
    print(f"#  Run mode: {RUN_MODE}")
    print(f"#  Results dir: {cfg['results_dir']}")
    print(f"#  Coverage levels: {cfg['coverage_levels']}")
    print(f"#  Vocab: {cfg['vocab_size']} (3-base eta with eta={cfg['eta']})")
    print(f"{'#'*78}")

    if not os.path.exists(cfg['dataset_path']):
        msg = (f"Dataset not found: {cfg['dataset_path']}.\n"
               f"  Run the 3-base eta dataset_generator notebook first with ERROR_MODEL='{profile_name}'.")
        print(f"  [ERROR] {msg}")
        batch_summary.append({'profile': profile_name, 'status': 'no dataset',
                              'dir': cfg['results_dir']})
        continue

    os.makedirs(cfg['results_dir'], exist_ok=True)
    SYMBOL_TO_IDX, IDEAL_VECTORS = build_symbol_to_idx_and_vectors(cfg)
    IDEAL_VECTORS = IDEAL_VECTORS.to(device)

    results = {
        'coverage': list(cfg['coverage_levels']),
        'per_M': {'lstm': [], 'mindist': [], 'kl': [], 'ml': []},
        'config': dict(cfg),
    }
    results['config']['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    all_histories = {}

    try:
        for M in cfg['coverage_levels']:
            metrics, history = run_experiment_for_coverage(
                M, cfg, SYMBOL_TO_IDX, IDEAL_VECTORS, device, RUN_MODE)
            for k in ['lstm','mindist','kl','ml']:
                results['per_M'][k].append(metrics[k])
            if history is not None:
                all_histories[M] = history

        mat_path  = os.path.join(cfg['results_dir'], 'experiment_results.mat')
        txt_path  = os.path.join(cfg['results_dir'], 'experiment_results.txt')
        plot_path = os.path.join(cfg['results_dir'], 'final_comparison_plot.png')

        save_results_mat(results, cfg, mat_path)
        save_results_txt(results, cfg, txt_path)
        plot_comparison(results, plot_path, cfg)
        if all_histories:
            with open(os.path.join(cfg['results_dir'], 'all_training_histories.json'), 'w') as f:
                json.dump({str(k): v for k, v in all_histories.items()}, f, indent=2)
        batch_summary.append({'profile': profile_name, 'status': 'ok',
                              'dir': cfg['results_dir']})
    except Exception as e:
        print(f"\n[ERROR] Profile {profile_name} failed: {type(e).__name__}: {e}")
        batch_summary.append({'profile': profile_name,
                              'status': f'FAILED: {type(e).__name__}',
                              'dir': cfg['results_dir']})

print(f"\n\n{'='*78}")
print(f" BATCH SUMMARY ({len(batch_summary)} profile(s), total {time.time()-batch_t0:.1f}s)")
print(f"{'='*78}")
print(f"   {'Profile':<8} {'Status':<22}   Output directory")
print(f"   {'-'*8} {'-'*22}   {'-'*40}")
for r in batch_summary:
    print(f"   {r['profile']:<8} {r['status']:<22}   {r['dir']}")
print(f"{'='*78}")




##############################################################################
#  [1/1]  Profile: EZ17  (Illumina miSeq + Twist Bioscience)
#  Run mode: train_and_eval
#  Results dir: ./Results/EZ17_eta3mix_0.1_n5_H128_L2_lr1e-03_eps0.01_seed42
#  Coverage levels: [50]
#  Vocab: 80 (3-base eta with eta=0.1)
##############################################################################

  COVERAGE M = 50  |  profile = EZ17  |  mode = train_and_eval
  seq_length = 136, vocab = 80, alphabet = eta3mix_0.1_n5
   Dataset loaded: 100,000 samples from ./dataset/dna_EZ17_eta3mix_0.1_n5_100000_50.pkl
      Saved train/val split to ./Results/EZ17_eta3mix_0.1_n5_H128_L2_lr1e-03_eps0.01_seed42/split_indices_M50.pkl
   Train: 80,000  |  Val: 20,000  |  split_seed=76354284
   Model: 80-class Bi-LSTM, 553,040 parameters
   Training: epochs=100, patience=10, warmup=10, lr=0.001 → 1e-06
   Epoch 001/100 | Train: 4.3561 | Val: 4.2539 | LR: 1.00e-04 | Time: 1391.7s
   Epoch 002/100 | Train: 3.9138 | Val

/DATA/shubham19/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:240: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 1.4439 | Val: 1.4176 | LR: 1.00e-03 | Time: 1368.6s
      Val loss improved (1.4197 -> 1.4176). Saving.
   Epoch 012/100 | Train: 1.4388 | Val: 1.4156 | LR: 1.00e-03 | Time: 1372.0s
      Val loss improved (1.4176 -> 1.4156). Saving.
   Epoch 013/100 | Train: 1.4352 | Val: 1.4134 | LR: 9.99e-04 | Time: 1371.5s
      Val loss improved (1.4156 -> 1.4134). Saving.
   Epoch 014/100 | Train: 1.4319 | Val: 1.4120 | LR: 9.97e-04 | Time: 1368.5s
      Val loss improved (1.4134 -> 1.4120). Saving.
   Epoch 015/100 | Train: 1.4293 | Val: 1.4091 | LR: 9.95e-04 | Time: 1369.4s
      Val loss improved (1.4120 -> 1.4091). Saving.
   Epoch 016/100 | Train: 1.4267 | Val: 1.4080 | LR: 9.92e-04 | Time: 1370.9s
      Val loss improved (1.4091 -> 1.4080). Saving.
   Epoch 017/100 | Train: 1.4246 | Val: 1.4066 | LR: 9.89e-04 | Time: 1372.4s
      Val loss improved (1.4080 -> 1.4066). Saving.
   Epoch 018/100 | Train: 1.4228 | Val: 1.4055 | LR: 9.85e-04 | Time: 1370.0s
      Val lo

/tmp/ipykernel_457421/4130657985.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_path, map_location=device))



   Results @ M=50:
      Decoder          Acc(%)        NER        SER   FailRate
      Bi-LSTM         47.6078   0.523922   1.000000   1.000000
      Min.Dist.       46.0415   0.539585   1.000000   1.000000
      KL Div.         46.3606   0.536394   1.000000   1.000000
      Max Lik.        46.3606   0.536394   1.000000   1.000000
   Saved .mat results: ./Results/EZ17_eta3mix_0.1_n5_H128_L2_lr1e-03_eps0.01_seed42/experiment_results.mat
   Saved .txt report : ./Results/EZ17_eta3mix_0.1_n5_H128_L2_lr1e-03_eps0.01_seed42/experiment_results.txt

 COMPOSITE-DNA DECODER — EXPERIMENT REPORT (3-base eta alphabet)

Timestamp        : 2026-05-22 04:36:15
Run mode         : train_and_eval
Error profile    : EZ17
Platform         : Illumina miSeq + Twist Bioscience
Alphabet mode    : eta3mix_0.1_n5
Eta              : 0.1
Vocabulary size  : 80
Sequence length  : 136
Dataset          : ./dataset/dna_EZ17_eta3mix_0.1_n5_100000_50.pkl

Hyperparameters
------------------------------------------------